# Machine Learning project in SoSe 2025 at HTW Saar
## Idea
The goal of this project is getting the genre(s) of a game trough its given metadata

## Dataset
For our project we use a Steam dataSet from kaggle. You can find it under the following URL: [Kaggle.com](https://www.kaggle.com/datasets/artermiloff/steam-games-dataset/data)

### Importing the dataSet
The dataSet is imported and added as a variable.

In [6]:
import numpy as np
import pandas as pd

# load data
# appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
dataset = pd.read_csv("./games_march2025_cleaned_10k.csv",sep=",")
print(dataset.head())

    appid                             name release_date  required_age  price  \
0     730                 Counter-Strike 2   2012-08-21             0   0.00   
1  578080              PUBG: BATTLEGROUNDS   2017-12-21             0   0.00   
2     570                           Dota 2   2013-07-09             0   0.00   
3  271590        Grand Theft Auto V Legacy   2015-04-13            17   0.00   
4  359550  Tom Clancy's Rainbow Six® Siege   2015-12-01            17   3.99   

   dlc_count                               detailed_description  \
0          1  For over two decades, Counter-Strike has offer...   
1          0  LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...   
2          2  The most-played game on Steam. Every day, mill...   
3          0  When a young street hustler, a retired bank ro...   
4          9  Edition Comparison Ultimate Edition The Tom Cl...   

                                      about_the_game  \
0  For over two decades, Counter-Strike has offer...   
1  L

## Preparation of the Training-Set
### Removing Uniques
We remove the following features from the Training-Set as they can uniquely identify a datapoint:
- AppId
- Name of the Game
- Realease Date
- Reviews
- Header Image
- Website
- Support URL
- Support Email
- MetaCritic URL
- Developer
- Publisher
- Screenshots
- Movies
- Estimated Owners

In [ ]:
# appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
dataset.drop(['appid', 'name', 'release_date', 'reviews', 'header_image', 'website', 'support_url', 'support_email',
              'metacritic_url', 'developers', 'publishers', 'screenshots', 'movies', 'estimated_owners'],
              axis=1, inplace=True)
print(dataset.head())

   required_age  price  dlc_count  \
0             0   0.00          1   
1             0   0.00          0   
2             0   0.00          2   
3            17   0.00          0   
4            17   3.99          9   

                                detailed_description  \
0  For over two decades, Counter-Strike has offer...   
1  LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...   
2  The most-played game on Steam. Every day, mill...   
3  When a young street hustler, a retired bank ro...   
4  Edition Comparison Ultimate Edition The Tom Cl...   

                                      about_the_game  \
0  For over two decades, Counter-Strike has offer...   
1  LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...   
2  The most-played game on Steam. Every day, mill...   
3  When a young street hustler, a retired bank ro...   
4  “One of the best first-person shooters ever ma...   

                                   short_description  \
0  For over two decades, Counter-Strike has off

### Structurize Text
**TODO: check if makes sense**
The dataset holds a lot of unstructured data, we use Term Frequency-Inverse Document Frequency to structurize most Text-Features.
It is important to use an new Instance for each feature so they don't overlap with each other. 

### Standardize Values
We standardize only the text features to remove the stop words. The dataset allready provides standardized numerical features.

In [9]:
from sklearn.compose import make_column_transformer
from sklearn.feature_extraction.text import TfidfVectorizer
# appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
column_transformer = make_column_transformer(
    (TfidfVectorizer(stop_words='english'), ['detailed_description']),
    (TfidfVectorizer(stop_words='english'), ['about_the_game']),
    (TfidfVectorizer(stop_words='english'), ['short_description']),
    ('passthrough', ['required_age','price','dlc_count','reviews','windows','mac','linux','metacritic_score','achievements','recommendations','notes','supported_languages','full_audio_languages','categories','genres','user_score','score_rank','positive','negative','average_playtime_forever','average_playtime_2weeks','median_playtime_forever','median_playtime_2weeks','discount','peak_ccu','tags','pct_pos_total','num_reviews_total','pct_pos_recent','num_reviews_recent'])
)

dataset = column_transformer.fit_transform(dataset)
print(dataset.head())

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 1 and the array at index 3 has size 9999


### Removing Bundles
**(TODO: decide whether yes or no), not as important as i thought**
As bundles don't have clear genre(s) defined (e.g. publisher bundles )

### Handling missing values
Removing NaN values in the dataSet and setting missing numerical feature values to the mean feature count. Missing Text values are set to a default String `Unknown`.

In [6]:
# Setting missing numeric values to the mean
dataset.fillna(dataset.mean(numeric_only=True), inplace=True)
# Setting missing text values to 'Unknown'
dataset.fillna('Unknown', inplace=True)
# Setting missing values in other columns to NaN
dataset.dropna(inplace=True)

# Data Split
Splitting our dataSet to training and testing data. The relation is 80% training and 20% testing data.

In [ ]:
from sklearn.model_selection import train_test_split

# Setting the target feature 'genres' and dropping it from the dataset
X = dataset.drop('genres', axis=1)
y = dataset['genres']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape}, Testing: {X_test.shape}")

Trainingsdaten: (7999, 33), Testdaten: (2000, 33)


# Model Selection
**TODO Deciding which model to use for this task**

### Training
**TODO Train the Selected Model with the training data**

# Evaluation
**TODO Test the Model with the test data**

# Optimization
**TODO optimize the model based on the test results**

# Validation
**TODO Predict actual values**

# Conclusion and outlook
**TODO Write a conclusion and outlook what can be done and where the issues were.**